# Building LLM Agents for Chemistry

In this notebook you will learn how **LLM agents** can use **tools** to answer chemistry questions that require real computation — not just pattern-matching from training data.



## What is an LLM Agent?

A plain language model can only generate text. When you ask it *"what is the molecular weight of aspirin?"* it may give a plausible-sounding number — but it is guessing from its training data, not computing anything.

An **agent** combines a language model with **tools**: Python functions the model can call to get exact, up-to-date answers. 

In this notebook we are not using a full agent framework like LangChain or LangGraph.
Instead, we implement a minimal tool‑calling protocol by hand:

    We tell the model which tools exist, and how to “ask” for them, using a special string format:
    TOOL:tool_name:ARGS.

    When the model outputs such a line, our Python code (the “orchestrator”) parses the tool name and arguments, calls the corresponding Python function, and then sends the tool result back to the model in a follow‑up prompt.
    
    This is the same idea that larger agent frameworks use internally; here we make it explicit so you can see how an LLM becomes an “agent” once it can request tools and use their results.
The following figure from [Haystack](https://haystack.deepset.ai/blog/introducing-haystack-agents) nicely illustrates the loop:

![Figure taken from HayStack (by deepset) illustrating the ReaAct loop.](https://haystack.deepset.ai/blog/introducing-haystack-agents/agents.png)

This is inspired by [chain-of-thought prompting](https://arxiv.org/abs/2201.11903), which has been shown to be effective in improving the performance of LLMs on a variety of tasks.

## Prerequisites
 We need to install first our tools: RDKit gives you instant “chemistry” tools; this is the same class of tool ChemCrow and ChemMCP wrap in more complex agents. 


In [ ]:
pip install rdkit-pypi google-genai langchain openai

## API keys
To be able to query the LLM we need an access to the model and API key. You can get a free API key at https://aistudio.google.com/ and start calling Gemini 2.5 Flash and related “Flash” models without billing enabled. Free tier limits are on the order of 10 requests/minute and 500 requests/day for Gemini 2.5 Flash, with a token budget (~250k tokens/minute) that is more than enough for our classroom demo or light development. You will need either an **OpenAI** API key or a **Google Gemini** API key (both are used in the examples below). Set them as environment variables: 

**NOTE**: The API key for OpenAi has some paid tokens and should be enough to run the demo. Please do not use it outside of the demo.  Let me know if it runs out of the tokens.   

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "" # Pass your Google Gemini API key here
os.environ["OPENAI_API_KEY"] = "" # Pass the API key for OPEN AI given in the instructions on MyCourses

## Instructions for the exercises:

You can do the exercies in Part 1 and Part 2. The exercieses will add up to 1.5 points. Or you can do the exercise in Part 3, it will count as 1.5 points.

# Part 1 — Molecular Properties with RDKit

## 1.1 The Chemistry Tools (provided — read, don't edit)

The two functions below are your **tools**. They wrap RDKit to compute two important drug-likeness descriptors:

| Tool | What it computes
|------|-----------------|
| `calc_mol_weight` | Molecular weight (g/mol) |
| `calc_logp` | cLogP (octanol/water partition)| 

Both accept a **SMILES string** — a compact text representation of a molecule's connectivity. For example, aspirin is `CC(=O)Oc1ccccc1C(=O)O`.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# ── TOOL 1 ──────────────────────────────────────────────────────────────────
def calc_mol_weight(smiles: str) -> float:
    """Return the molecular weight (g/mol) for a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return Descriptors.MolWt(mol)

# ── TOOL 2 ──────────────────────────────────────────────────────────────────
def calc_logp(smiles: str) -> float:
    """Return the Crippen cLogP for a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return Descriptors.MolLogP(mol)

# Quick sanity check — aspirin (MW ≈ 180 g/mol, logP ≈ 1.31)
aspirin = "CC(=O)Oc1ccccc1C(=O)O"
print(f"Aspirin MW   : {calc_mol_weight(aspirin):.2f} g/mol")
print(f"Aspirin cLogP: {calc_logp(aspirin):.2f}")

## 1.2 Telling the LLM About the Tools — the Tool Prompt

The model cannot see your Python code. You must describe what tools exist and **exactly how to call them** inside the system prompt. This description is called the **tool documentation** (or *tool doc*).

Key design decisions here:
* Assign each tool a **unique name** the model must use.
* Specify the **output format** precisely so you can parse it with a simple `split`.
* Use `IMPORTANT RULES` to stop the model from guessing when it should be computing.

In [ ]:
TOOL_DOC_PART1 = """
You have access to two tools:

1) tool name : calc_mol_weight
   description: Compute the molecular weight (in g/mol) from a SMILES string.

2) tool name : calc_logp
   description: Compute the cLogP (octanol/water partition coefficient) from a SMILES string.

IMPORTANT RULES:
- For ANY question asking for a NUMERICAL molecular property (weight, logP, size, etc.)
  you MUST call the appropriate tool instead of guessing.
- When you call a tool, the tool call must be the ONLY thing in your reply.
  Do NOT add any text before or after it.

FORMAT:
  TOOL:calc_mol_weight:SMILES_HERE
  TOOL:calc_logp:SMILES_HERE
"""

## 1.3 Setting Up the LLM (Gemini)

We use Google Gemini here, but the agent pattern is identical for any chat model. The `gemini_chat` helper sends a single prompt and returns the model's text reply.

In [ ]:
import os
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
# we define our chat function here
def gemini_chat(prompt: str) -> str:
    response = client.models.generate_content(
        model="gemini-flash-latest",  # free-tier  model
        contents=prompt,
    )
    return response.text
    
# Smoke test
print(gemini_chat("In one sentence: what is a SMILES string?"))

## 1.4 The Agent Loop

The agent runs in two LLM calls:

1. **First call** — give the model the user question + tool doc. It either answers directly or emits a `TOOL:…` line.
2. **Tool execution** — if a tool call was detected, Python runs the matching function.
3. **Second call** — feed the numeric result back to the model so it can write a clear, natural-language answer.


In [ ]:
def agent_first_turn_part1(user_prompt: str) -> str:
    """First LLM call: decide whether to answer or call a tool."""
    system_prompt = f"""You are a helpful chemistry assistant.
You can answer general conceptual questions directly.
BUT for any question that needs a numeric molecular property you MUST call a tool.
{TOOL_DOC_PART1}
User: {user_prompt}
"""
    return gemini_chat(system_prompt).strip()


def run_agent_part1(user_prompt: str) -> str:
    """Full agent loop for Part 1."""
    # ── Step 1: first LLM call ───────────────────────────────────────────────
    first_reply = agent_first_turn_part1(user_prompt)

    # ── Step 2: detect tool call ─────────────────────────────────────────────
    tool_line = None
    for line in first_reply.splitlines():
        if line.strip().startswith("TOOL:"):
            tool_line = line.strip()
            break

    if tool_line is None:          # model answered without a tool
        return first_reply

    # ── Step 3: parse tool call ──────────────────────────────────────────────
    try:
        _, tool_name, arg = tool_line.split(":", 2)
    except ValueError:
        return f"(Malformed tool call) {tool_line}"

    # ── Step 4: execute the tool ─────────────────────────────────────────────
    try:
        if tool_name == "calc_mol_weight":
            value = calc_mol_weight(arg)
            result_text = f"The result of calc_mol_weight({arg}) is {value:.2f} g/mol."
        elif tool_name == "calc_logp":
            value = calc_logp(arg)
            result_text = f"The result of calc_logp({arg}) is {value:.2f}."
        else:
            return f"(Unknown tool) {tool_line}"
    except Exception as e:
        followup = f"""The tool {tool_name} failed: {e}
User question: {user_prompt}
Explain what went wrong and ask the user to check their input."""
        return gemini_chat(followup)

    # ── Step 5: second LLM call — explain the result ─────────────────────────
    followup = f"""You called a chemistry tool to answer a user question.
User question : {user_prompt}
Tool output   : {result_text}
Write a clear answer in 2–4 sentences. Include the SMILES used, the number, and a brief explanation."""
    return gemini_chat(followup)


# ── Demo ─────────────────────────────────────────────────────────────────────
print(run_agent_part1("What is the molecular weight of aspirin?"))
print("---")
print(run_agent_part1("Is caffeine more lipophilic than aspirin?"))

# Student Exercise 1 — Add a New Property Tool (0.5 points)

The tool `calc_num_hbd` below computes the **number of hydrogen bond donors**. Your task is to **wire it into the agent** so the model can use it.

**Steps:**
1. Run the cell below to see what the tool returns for aspirin.
2. Copy `TOOL_DOC_PART1` into a new variable `TOOL_DOC_EX1` and add a description for `calc_num_hbd`.
3. Add the `calc_num_hbd` branch inside a copy of `run_agent_part1` called `run_agent_ex1`.
4. Test it with: *"How many hydrogen bond donors does ibuprofen have?"* (SMILES: `CC(C)Cc1ccc(cc1)C(C)C(=O)O`)

In [ ]:
# ── PROVIDED TOOL — do not modify ────────────────────────────────────────────
from rdkit.Chem import Lipinski

def calc_num_hbd(smiles: str) -> int:
    """Return the number of hydrogen bond donors for a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return Lipinski.NumHDonors(mol)

# Quick test
print("Aspirin HBD:", calc_num_hbd("CC(=O)Oc1ccccc1C(=O)O"))  # expect 1

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 1. Create TOOL_DOC_EX1 (copy of TOOL_DOC_PART1, extended with calc_num_hbd)
TOOL_DOC_EX1 = """
# TODO: copy the tool doc above and add the new tool description
"""

# 2. Create run_agent_ex1 (copy of run_agent_part1 with the new tool branch)
def run_agent_ex1(user_prompt: str) -> str:
    # TODO
    pass

# 3. Test
# print(run_agent_ex1("How many hydrogen bond donors does ibuprofen (CC(C)Cc1ccc(cc1)C(C)C(=O)O) have?"))

# Part 2 — 3D Structure Analysis with ASE

## 2.1 The Structure Tools (provided — read, don't edit)

This section works with **XYZ files** — a simple text format listing every atom's element and 3D coordinates. We use the **Atomic Simulation Environment (ASE)** to read and analyse them.

Three tools are provided:

| Tool | Arguments | Returns |
|------|-----------|--------|
| `load_structure` | `name`, `filename` | Loads an XYZ file into memory under a given name |
| `bond_distance` | `struct_name`, `i`, `j` | Distance in Å between atoms *i* and *j* (0-based) |
| `center_of_mass` | `struct_name` | (x, y, z) centre of mass in Å |

Structures are stored in a shared dictionary so every tool can access them by name after loading.

In [ ]:
import numpy as np
from ase.io import read

# Shared in-memory store — all tools read/write here
structures = {}

# ── TOOL 1 ──────────────────────────────────────────────────────────────────
def load_structure(name: str, filename: str) -> str:
    """Load an XYZ file from disk and store it under `name`."""
    atoms = read(filename)
    structures[name] = atoms
    return f"Loaded structure '{name}' with {len(atoms)} atoms."

# ── TOOL 2 ──────────────────────────────────────────────────────────────────
def bond_distance(struct_name: str, i: int, j: int) -> float:
    """Return the distance in Å between atoms i and j (0-based) in struct_name."""
    atoms = structures.get(struct_name)
    if atoms is None:
        raise ValueError(f"Unknown structure: {struct_name!r}. Call load_structure first.")
    return float(np.linalg.norm(atoms[i].position - atoms[j].position))

# ── TOOL 3 ──────────────────────────────────────────────────────────────────
def center_of_mass(struct_name: str) -> tuple:
    """Return the centre of mass (x, y, z) in Å for struct_name."""
    atoms = structures.get(struct_name)
    if atoms is None:
        raise ValueError(f"Unknown structure: {struct_name!r}. Call load_structure first.")
    com = atoms.get_center_of_mass()
    return tuple(float(x) for x in com)

print("Tools ready: load_structure, bond_distance, center_of_mass")

## 2.2 Setting Up the LLM (OpenAI)

For Part 2 we switch to OpenAI's `gpt-4o-mini`. The multi-turn message format lets us keep context across the two agent calls more explicitly.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def openai_chat(messages: list, temperature: float = 0.2) -> str:
    """Send a list of role/content dicts to gpt-4o-mini and return the text."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

# Smoke test
print(openai_chat([{"role": "user", "content": "In one sentence: what is an XYZ file?"}]))

## 2.3 Tool Documentation for Part 2

Notice a new rule: the model must **load a structure before querying it**. This is an example of **tool ordering** — some tools have prerequisites. Writing clear rules in the tool doc is how you enforce this.

In [ ]:
TOOL_DOC_PART2 = """
You have access to three tools for analysing 3D molecular structures.

1) tool name : load_structure
   description: Load an XYZ file from disk and store it under a given name.
   format     : TOOL:load_structure:STRUCT_NAME,FILENAME

2) tool name : bond_distance
   description: Compute the distance (Å) between atoms i and j (0-based index)
                in a previously loaded structure.
   format     : TOOL:bond_distance:STRUCT_NAME,i,j

3) tool name : center_of_mass
   description: Compute the centre of mass (x, y, z in Å) of a loaded structure.
   format     : TOOL:center_of_mass:STRUCT_NAME

IMPORTANT RULES:
- A structure MUST be loaded before bond_distance or center_of_mass can use it.
- When calling a tool, your entire reply MUST be a single TOOL: line. No other text.
- Do NOT claim you cannot access files — use load_structure to load them.
"""

In [ ]:
def run_agent_part2(user_prompt: str, verbose: bool = False) -> str:
    """Agent loop for 3D-structure questions."""
    # ── Step 1: first LLM call ───────────────────────────────────────────────
    messages = [
        {"role": "system",
         "content": f"You are a helpful molecular modelling assistant.\n{TOOL_DOC_PART2}"},
        {"role": "user", "content": user_prompt},
    ]
    first_reply = openai_chat(messages, temperature=0.1)
    if verbose:
        print("[LLM turn 1]:", repr(first_reply))

    # ── Step 2: detect tool call ─────────────────────────────────────────────
    tool_line = None
    for line in first_reply.splitlines():
        if line.strip().startswith("TOOL:"):
            tool_line = line.strip()
            break

    if tool_line is None:
        return first_reply

    # ── Step 3: parse tool call ──────────────────────────────────────────────
    try:
        _, tool_name, arg_str = tool_line.split(":", 2)
    except ValueError:
        return f"(Malformed tool call) {tool_line}"

    # ── Step 4: execute the tool ─────────────────────────────────────────────
    try:
        if tool_name == "load_structure":
            struct_name, filename = [x.strip() for x in arg_str.split(",", 1)]
            result_text = load_structure(struct_name, filename)
        elif tool_name == "bond_distance":
            struct_name, i_str, j_str = [x.strip() for x in arg_str.split(",")]
            dist = bond_distance(struct_name, int(i_str), int(j_str))
            result_text = f"bond_distance({struct_name}, {i_str}, {j_str}) = {dist:.3f} Å"
        elif tool_name == "center_of_mass":
            struct_name = arg_str.strip()
            com = center_of_mass(struct_name)
            result_text = (f"center_of_mass({struct_name}) = "
                           f"({com[0]:.3f}, {com[1]:.3f}, {com[2]:.3f}) Å")
        else:
            return f"(Unknown tool) {tool_line}"
    except Exception as e:
        followup = [
            {"role": "system", "content": "A tool call failed."},
            {"role": "user",
             "content": f"User question: {user_prompt}\nError: {e}\nExplain and ask for clarification."},
        ]
        return openai_chat(followup)

    # ── Step 5: second LLM call — explain the result ─────────────────────────
    followup = [
        {"role": "system", "content": "You called a molecular modelling tool and got a result."},
        {"role": "user",
         "content": f"User question: {user_prompt}\nTool output: {result_text}\n"
                    "Answer in 2–4 sentences: what you did and what the result means."},
    ]
    return openai_chat(followup, temperature=0.3)


# ── Demo ─────────────────────────────────────────────────────────────────────
print(run_agent_part2("Load the file water.xyz and call it 'water'."))
print("---")
print(run_agent_part2("What is the O–H bond length in 'water' (atoms 0 and 1)?"))

# Student Exercise 2 — Add an Element-Counting and Bonding Analysis Tool (1.0 points)

The provided tool `count_element`counts how many atoms of a given element are in a loaded structure, while `analyze_oxygen_bonding` classifies the bonding environments for O atoms.

**Steps:**
1. Test `count_element` and `analyze_oxygen_bonding` directly on the privided amorphous carbon structure 'aCHO.xyz'.
2. Copy `TOOL_DOC_PART2` into `TOOL_DOC_EX2` and add an entry for `count_element` as well as `analyze_oxygen_bonding`.
3. Create `run_agent_ex2` by extending `run_agent_part2` with the new branch.
4. Test: *"How many hydrogen, oxygen and carbon atoms are in the provided amorphous carbon structure 'aCHO.xyz'?"*

In [ ]:
# ── PROVIDED TOOLs — do not modify ────────────────────────────────────────────
def count_element(struct_name: str, element: str) -> int:
    """Count atoms of `element` (e.g. 'H', 'O') in a loaded structure."""
    atoms = structures.get(struct_name)
    if atoms is None:
        raise ValueError(f"Unknown structure: {struct_name!r}.")
    return sum(1 for a in atoms if a.symbol == element)
    
def analyze_oxygen_bonding(struct_name: str) -> str:
    """
    Analyze O atoms in a stored structure:
    - Count how many O atoms.
    - For each O, count neighbors and classify simple bonding environments.

    Returns a human-readable summary string.
    """
    atoms = structures.get(struct_name)
    if atoms is None:
        raise ValueError(f"Unknown structure: {struct_name}")

    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()

    o_indices = [i for i, s in enumerate(symbols) if s == "O"]

    if not o_indices:
        return f"Structure '{struct_name}' contains no oxygen atoms."

    # Simple cutoff-based "bond" definition (Å)
    cutoff = 1.8  # works OK for O–H and many O–X bonds

    summaries = []
    for i in o_indices:
        pos_i = positions[i]
        neighbors = []
        for j, (pos_j, sym_j) in enumerate(zip(positions, symbols)):
            if j == i:
                continue
            dist = float(np.linalg.norm(pos_i - pos_j))
            if dist <= cutoff:
                neighbors.append((j, sym_j, dist))

        # Simple classification based on neighbor count and types
        neighbor_symbols = [sym for (_, sym, _) in neighbors]
        n_neighbors = len(neighbors)

        if n_neighbors == 0:
            env = "isolated O (no neighbors within cutoff)"
        elif neighbor_symbols.count("H") >= 2:
            env = "water-like or hydroxyl O (bonded to ≥2 H)"
        elif "C" in neighbor_symbols and n_neighbors == 1:
            env = "carbonyl-like O (bonded to 1 C neighbor)"
        elif "C" in neighbor_symbols and n_neighbors >= 2:
            env = "bridging or multi-coordinated O (bonded to C and others)"
        else:
            env = f"O with {n_neighbors} neighbors: {', '.join(neighbor_symbols)}"

        # Short per-atom summary
        neigh_str = ", ".join(
            f"{sym}{idx}({dist:.2f} Å)" for idx, sym, dist in neighbors
        ) or "none"
        summaries.append(
            f"O{ i } environment: {env}. Neighbors: {neigh_str}."
        )

    header = f"Structure '{struct_name}' has {len(o_indices)} oxygen atom(s)."
    return header + "\n" + "\n".join(summaries)

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 1. Create TOOL_DOC_EX2 (copy of TOOL_DOC_PART2, extended with count_element and analyze_oxygen_bonding)
TOOL_DOC_EX2 = """
# TODO: copy the tool doc above and add the new tools description
"""

# 2. Create run_agent_ex2 (copy of run_agent_part2 with the new tool branch)
def run_agent_ex2(user_prompt: str,
                    max_steps: int = 10,
                    verbose: bool = False) -> str:
    # TODO
    # Hint you need to add a loop. The loop is what enables the agent to use multiple tools sequentially instead of stopping after one tool call. The example loop is given here: 
    # for step in range(max_steps):

    #     # Ask model
    #     reply = openai_chat(messages, temperature=0.1)

    #     if verbose:
    #         print(f"[Step {step+1}] {reply}")

    #     # Final answer → stop
    #     if not reply.startswith("TOOL:"):
    #         return reply

    #     # Parse tool call
    #     _, tool_name, arg_str = reply.split(":", 2)

    #     # Execute tool
    #     try:
    
    pass
# print(run_agent_ex3(
#     """
#     Load 'aCHO.xyz' and call it 'ac'.

#     Then:
#     1. Check how many carbon, hydrogen, and oxygen atoms are present.
#     2. If oxygen exists, analyse oxygen bonding.
#     3. Summarise the local oxygen environments.
#     4. Check center of mass
#     5. What is bond distance between O atom and H atom 
#     """
# ))

**Visualize the structure**

In [ ]:
!pip install weas_widget
from weas_widget import WeasWidget
atoms = read("aCHO.xyz", index=":") # run with index="-1" so see the last snapshot only
viewer = WeasWidget()
viewer.avr.model_style = 1 # ball & stick mode
viewer.from_ase(atoms)
viewer.avr.show_bonded_atoms = True # show bonds across periodic boundaries
print("The database has:", len(atoms), " CH structures")
viewer

if you cannot see the structure you need to refresh the browser to see the plot. 

# Student Exercise 3 — Design Your Own Tool (1.5 points, you can do only this exercise )

Propose your own tools and implement it end-to-end: write the tool, add it to the tool doc, integrate it into the agent, and show a demo query.

For whichever option you choose:
1. Write and test the Python function.
2. Add it to a tool doc string.
3. Integrate it into an agent (you can copy any run_agent function from above).
4. Run at least two demo queries.